Cell 1: Setup & MLflow Experiment

In [0]:
# MAGIC %md
# MAGIC # ⚡ EV Site Selection — NB03: Scoring + MLflow
# MAGIC **Layer:** Gold | **Objective:** คะแนนทำเลด้วย 3 กลยุทธ์, K-Means Clustering และ Register Best Model

import mlflow
import mlflow.spark
from pyspark.sql.functions import col, desc, lit
import pandas as pd

DATABASE_NAME = "ev_site_selection"
spark.sql(f"USE {DATABASE_NAME}")

# ปรับ path ให้ตรงกับ Workspace ของคุณ
experiment_name = "/ev_site_selection_bkk_v2"
mlflow.set_experiment(experiment_name)

print(f"✅ MLflow Experiment: {experiment_name}")


2026/05/12 00:06:24 INFO mlflow.tracking.fluent: Experiment with name '/ev_site_selection_bkk_v2' does not exist. Creating a new experiment.


✅ MLflow Experiment: /ev_site_selection_bkk_v2


Cell 2: Load Silver Data & Define Scoring Function

In [0]:
# MAGIC %md
# MAGIC ## Cell 2: Load Silver Data & Define Scoring Function

import mlflow
import math
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, desc, lit

print("⏳ Loading Silver Data...")
df_silver = spark.table("silver_ev_features")

if "cnt_charging_station" not in df_silver.columns:
    df_silver = df_silver.withColumn("cnt_charging_station", lit(0.0))

assembler   = VectorAssembler(
    inputCols=["pop_density", "cnt_charging_station", "flood_occurrence"],
    outputCol="features_raw"
)
df_assembled = assembler.transform(df_silver)
scaler       = MinMaxScaler(inputCol="features_raw", outputCol="features_scaled")
df_scaled    = scaler.fit(df_assembled).transform(df_assembled)

df_ready = df_scaled \
    .withColumn("features_arr",    vector_to_array("features_scaled")) \
    .withColumn("norm_demand",     col("features_arr")[0]) \
    .withColumn("norm_competitor", col("features_arr")[1]) \
    .withColumn("norm_risk",       col("features_arr")[2]) \
    .select("grid_lat", "grid_lon", "norm_demand", "norm_competitor", "norm_risk")


def run_scoring_experiment(run_name, w_demand, w_competitor, w_risk):
    """Score grid cells, run K-Means, log metrics and artifacts to MLflow"""
    with mlflow.start_run(run_name=run_name) as run:
        print(f"🚀 Starting Run: {run_name}")

        mlflow.log_param("w_demand",     w_demand)
        mlflow.log_param("w_competitor", w_competitor)
        mlflow.log_param("w_risk",       w_risk)

        df_scored = df_ready.withColumn("total_score",
            (col("norm_demand")     * w_demand)     -
            (col("norm_competitor") * w_competitor) -
            (col("norm_risk")       * w_risk)
        )

        vec_assembler    = VectorAssembler(
            inputCols=["grid_lat", "grid_lon", "total_score"],
            outputCol="clustering_features"
        )
        df_cluster_ready = vec_assembler.transform(df_scored)
        model            = KMeans(k=3, seed=42,
                                  featuresCol="clustering_features",
                                  predictionCol="cluster").fit(df_cluster_ready)
        df_clustered     = model.transform(df_cluster_ready)

        evaluator = ClusteringEvaluator(
            featuresCol="clustering_features",
            predictionCol="cluster",
            metricName="silhouette",
            distanceMeasure="squaredEuclidean"
        )
        silhouette = evaluator.evaluate(df_clustered)
        mlflow.log_metric("silhouette_score", silhouette)

        top_20   = df_clustered.orderBy(desc("total_score")).limit(20).toPandas()
        csv_path = f"/tmp/top20_{run_name.replace(' ', '_')}.csv"
        top_20.to_csv(csv_path, index=False)
        mlflow.log_artifact(csv_path)

        avg_gap = 0
        if len(top_20) > 1:
            total_dist, count = 0, 0
            for i in range(len(top_20)):
                for j in range(i + 1, len(top_20)):
                    dist        = math.sqrt(
                        (top_20.iloc[i]["grid_lat"] - top_20.iloc[j]["grid_lat"]) ** 2 +
                        (top_20.iloc[i]["grid_lon"] - top_20.iloc[j]["grid_lon"]) ** 2
                    )
                    total_dist += dist
                    count      += 1
            avg_gap = total_dist / count if count > 0 else 0

        mlflow.log_metric("top20_avg_gap_score", avg_gap)
        print(f"   ✅ Silhouette: {silhouette:.4f} | Avg Gap: {avg_gap:.4f}")

        return df_clustered, model, run.info.run_id

print("✅ Silver data loaded. Scoring function ready.")


⏳ Loading Silver Data...
✅ Silver data loaded. Scoring function ready.


Cell 3: Run A — Aggressive Demand

In [0]:
# MAGIC %md
# MAGIC ## Cell 3: Run A — Aggressive Demand
# MAGIC เน้นประชากรสูง (w_demand=0.80, w_competitor=0.10, w_risk=0.10)

df_run_a, model_a, run_id_a = run_scoring_experiment(
    run_name="Run A: Aggressive Demand",
    w_demand=0.80, w_competitor=0.10, w_risk=0.10
)

🚀 Starting Run: Run A: Aggressive Demand
   ✅ Silhouette: 0.4954 | Avg Gap: 0.0936


Cell 4: Run B — Blue Ocean

In [0]:
# MAGIC %md
# MAGIC ## Cell 4: Run B — Blue Ocean
# MAGIC เน้นพื้นที่ไม่มีคู่แข่ง (w_demand=0.40, w_competitor=0.50, w_risk=0.10)

df_run_b, model_b, run_id_b = run_scoring_experiment(
    run_name="Run B: Blue Ocean",
    w_demand=0.40, w_competitor=0.50, w_risk=0.10
)

🚀 Starting Run: Run B: Blue Ocean
   ✅ Silhouette: 0.5260 | Avg Gap: 0.0944


Cell 5: Run C — Safe Play 

In [0]:
# MAGIC %md
# MAGIC ## Cell 5: Run C — Safe Play
# MAGIC เน้นพื้นที่ปลอดภัยจากน้ำท่วม (w_demand=0.40, w_competitor=0.20, w_risk=0.40)

df_run_c, model_c, run_id_c = run_scoring_experiment(
    run_name="Run C: Safe Play",
    w_demand=0.40, w_competitor=0.20, w_risk=0.40
)

🚀 Starting Run: Run C: Safe Play
   ✅ Silhouette: 0.5920 | Avg Gap: 0.0944


Cell 6: Compare Runs & Register Best Model

In [0]:
# MAGIC %md
# MAGIC ## Cell 6: Compare Runs & Register Best Model

from mlflow.models.signature import infer_signature
from pyspark.sql.functions import desc

print("🔍 Comparing MLflow runs...")
runs_df = mlflow.search_runs(experiment_names=[experiment_name])
runs_df["custom_eval"] = (
    runs_df["metrics.silhouette_score"] +
    runs_df["metrics.top20_avg_gap_score"] * 10
)
best_run      = runs_df.sort_values("custom_eval", ascending=False).iloc[0]
best_run_id   = best_run["run_id"]
best_run_name = best_run["tags.mlflow.runName"]
print(f"🏆 Best Run: {best_run_name} (run_id: {best_run_id})")

if   best_run_name == "Run A: Aggressive Demand": final_df, final_model = df_run_a, model_a
elif best_run_name == "Run B: Blue Ocean":         final_df, final_model = df_run_b, model_b
else:                                              final_df, final_model = df_run_c, model_c

sample_input  = final_df.drop("cluster").limit(5).toPandas()
sample_output = final_df.select("cluster").limit(5).toPandas()
signature     = infer_signature(sample_input, sample_output)

# สร้าง UC Volume สำหรับ MLflow tmp path
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
volume_name     = "mlflow_tmp_volume"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {current_catalog}.{DATABASE_NAME}.{volume_name}")
uc_tmp_path = f"/Volumes/{current_catalog}/{DATABASE_NAME}/{volume_name}"

with mlflow.start_run(run_id=best_run_id):
    mlflow.spark.log_model(
        spark_model=final_model,
        artifact_path="kmeans_ev_model",
        signature=signature,
        registered_model_name="EV_Site_Selection_KMeans",
        dfs_tmpdir=uc_tmp_path
    )
print("✅ Model registered to MLflow Registry")

final_gold_df = final_df.select(
    "grid_lat", "grid_lon", "norm_demand", "norm_competitor",
    "norm_risk", "total_score", "cluster"
)
final_gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DATABASE_NAME}.gold_grid_scored")

print(f"📦 gold_grid_scored: {final_gold_df.count():,} records saved")
display(final_gold_df.orderBy(desc("total_score")).limit(10))


🔍 Comparing MLflow runs...
🏆 Best Run: Run C: Safe Play (run_id: b41aa72d4c2543f08d7bb28c2e5fc906)


2026/05/12 00:07:59 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-va

Uploading artifacts:   0%|          | 0/22 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workspace.ev_site_selection.ev_site_selection_kmeans': https://dbc-22971c56-2ef2.cloud.databricks.com/explore/data/models/workspace/ev_site_selection/ev_site_selection_kmeans/version/3?o=7474648823980063


✅ Model registered to MLflow Registry
📦 gold_grid_scored: 18,492 records saved


grid_lat,grid_lon,norm_demand,norm_competitor,norm_risk,total_score,cluster
13.774,100.6376,1.0,0.0,0.0,0.4,0
13.7695,100.6376,1.0,0.0,0.0,0.4,0
13.7605,100.6478,0.8524789178775156,0.0,0.0,0.34099156715100626,0
13.7605,100.6427,0.8524789178775156,0.0,0.0,0.34099156715100626,0
13.7695,100.6274,0.8480114682133838,0.0,0.0,0.3392045872853535,0
13.7695,100.6325,0.8480114682133838,0.0,0.0,0.3392045872853535,0
13.774,100.6274,0.8480114682133838,0.0,0.0,0.3392045872853535,0
13.774,100.6325,0.8480114682133838,0.0,0.0,0.3392045872853535,0
13.756,100.6121,0.8199865705970585,0.0,0.0,0.3279946282388234,2
13.7515,100.6121,0.8199865705970585,0.0,0.0,0.3279946282388234,2
